# Submission Template: Module 2 Lab 2: Metric Selection Under Cost Pressure

Name: Alma Dollente  
Date: 10/31/2026

Fill this in as you work. Sections marked PREDICT must be written before you run the step.

---

## 0. Your unique values

These make your submission yours. Fill them in from your own account and your own run.

- **Which scenario you computed by hand:** Enterprise spam filter
  
- **Your total error cost for each of the four scenarios:**   
S1, Hospital sepsis screening Total Error Cost:   `$500,008` 
S4, Loan pre-approval triage Total Error Cost:    `$343,000`
S3, Warehouse part defect scan Total Error Cost:  `$27,140`
S2, Enterprise spam filter Total Error Cost:       `$7520`

- **The threshold recommendation you made for scenario S4:**  
Setting a step 95 threshold gives a zero-default policy, saving `$190,000` more than a baseline at step 0 threshold.

## 1. Predictions

Write the prediction BEFORE running. Then run, record the actual, and explain the gap. A wrong prediction reasoned honestly earns full credit.

### Step 1: Before you compute anything, look only at the four counts for your chosen scenario and predict: will accuracy be higher or lower than recall, and by roughly how much?

**My prediction (written before running):**

> I predict accuracy will be higher than recall

**What actually happened:**

> Enterprise spam filter Accuracy = TP + TN / everything  
> Enterprise spam filter Recall = TP / (TP + FP)  
> Enterprise spam filter precision = TP / (TP + FP)  
> Enterprise spam filter f1 = 2PR / (P + R)  

```
Accuracy = (1200 + 48000) / (1200 + 95 + 310 + 48000) 
         ≈ 0.991836

Recall = 1200 / (1200 + 310)
       ≈ 0.794702

Precision = 1200 / (1200 + 95)
          ≈ 0.926641

F1 = 2(0.926641)(0.794702) / (0.926641 + 0.79470)
   ≈ 0.855615
```

**Why the difference, or why I was right:**

> I was right because I estimated based on accuracy's calculation having a larger numerator than recall's numerator.  

In [46]:
### Step 2: Load the file and compute all four in code
import pandas as pd
df = pd.read_csv('m2_metric_scenarios.csv')

df['my_accuracy']  = (df['true_positive'] + df['true_negative']) / (df['true_positive'] + df['true_negative'] + df['false_positive'] + df['false_negative'])
df['my_precision'] = df['true_positive'] / (df['true_positive'] + df['false_positive'])
df['my_recall']    = df['true_positive'] / (df['true_positive'] + df['false_negative'])
df['my_f1']        = 2* (df['my_precision']*df['my_recall']) / (df['my_precision'] + df['my_recall'])

# check your work against the columns already in the file
print(df[['scenario_id','accuracy','my_accuracy',
          'recall','my_recall']])

  scenario_id  accuracy  my_accuracy  recall  my_recall
0          S1    0.9900     0.990000  0.9524   0.952381
1          S2    0.9918     0.991836  0.7947   0.794702
2          S3    0.9930     0.992986  0.9226   0.922619
3          S4    0.9536     0.953560  0.8455   0.845528


### Step 3: Before you run it, predict: will the scenario with the highest accuracy also have the lowest total error cost?

**My prediction (written before running):**
> The highest accuracy belongs to Warehouse part defect scan. I predict it will not have the lowest total error cost because accuracy denominator totals all cases in it's cost matrix outcome, while the total error cost is calculated with multiplying and adding it's scenario's values.  

**What actually happened:**
> Warehouse part defect scan had the highest accuracy 0.9930, but did not have the lowest total error cost `$27140`. S1, Hospital sepsis screening, had the highest total error cost.  

**Why the difference, or why I was right:**
>  I was right because comparing at the formulas, accuracy considers correct and incorrect predictions and divides all cases in it's cost matrix outcome, resulting in a smaller value. On the other hand, total cost error totals the it's multiplied error with its cost.  

In [47]:
df['total_error_cost'] = (df['false_positive'] * df['dollar_cost_per_false_positive']) + (df['false_negative'] * df['dollar_cost_per_false_negative']) 

print(df[['scenario_id','accuracy','total_error_cost']]
      .sort_values('total_error_cost', ascending=False))

  scenario_id  accuracy  total_error_cost
0          S1    0.9900            500008
3          S4    0.9536            343000
2          S3    0.9930             27140
1          S2    0.9918              7520


## 2. Break it on purpose
Paste the exact error text. Not a description of it.

In [48]:
s2_swap_cost_errors = df['scenario_id'] == 'S2'
df.loc[s2_swap_cost_errors, ['dollar_cost_per_false_positive', 'dollar_cost_per_false_negative']] = df.loc[s2_swap_cost_errors, ['dollar_cost_per_false_negative', 'dollar_cost_per_false_positive']].values
df['total_error_cost'] = (df['false_positive'] * df['dollar_cost_per_false_positive']) + (df['false_negative'] * df['dollar_cost_per_false_negative'])

pd.set_option('expand_frame_repr', False)
print(df[['scenario_id','dollar_cost_per_false_positive','dollar_cost_per_false_negative', 'total_error_cost']].sort_values('total_error_cost', ascending=False))


  scenario_id  dollar_cost_per_false_positive  dollar_cost_per_false_negative  total_error_cost
0          S1                               1                          250000            500008
3          S4                             300                            3200            343000
2          S3                              85                             900             27140
1          S2                              12                              40             13540


### Break 1: Swap the two cost columns for scenario S2 so a false positive costs what a false negative did, then recompute total error cost and the metric call.

**Exact error or behavior I observed:**

```
1          S2                              12                              40              7520
```

**What this tells me about how it works:**
> swapping one row's feature to get new accurate results must be reassigned to total_error_cost and recalculated to get an accurate result before the print statement, I needed to write the forulma again to compute correctly with the swapped rows.
> The new calculation with the swapped rows is:
```
1          S2                              12                              40             13540
```

### Break 2: Set all four scenarios to have equal false positive and false negative costs, then re rank them by total error cost.

In [49]:
df['dollar_cost_per_false_positive'] = df['dollar_cost_per_false_negative'] = 1
df['total_error_cost'] = (df['false_positive'] * df['dollar_cost_per_false_positive']) + (df['false_negative'] * df['dollar_cost_per_false_negative'])

print(df[['scenario_id', 'false_positive', 'false_negative', 'total_error_cost']].sort_values('total_error_cost', ascending=False))

  scenario_id  false_positive  false_negative  total_error_cost
1          S2              95             310               405
3          S4             130              95               225
2          S3              44              26                70
0          S1               8               2                10


**Exact error or behavior I observed:**

```
  scenario_id  false_positive  false_negative  total_error_cost
1          S2              95             310               405
3          S4             130              95               225
2          S3              44              26                70
0          S1               8               2                10
```

**What this tells me about how it works:**

> When errors are priced equally, the total_error_cost only accounts for the number of errors not the financial weight of the error.

# A simple sweep. 
**At each step you accept 4 more false positives in exchange for 1 fewer false negative.**  
Take scenario S4, the loan pre approval triage. Assume you can move the classification threshold and that doing so trades false positives against false negatives roughly one for four.  
Model at least five threshold settings and find the one that minimizes total error cost

In [50]:
fp, fn = 130, 95
cost_fp, cost_fn = 300, 3200
for step in range(0, 6):
    f_p = fp + 4*step
    f_n = fn - step
    total = (f_p * cost_fp) + (f_n * cost_fn)
    print(step, f_p, f_n, total)

0 130 95 343000
1 134 94 341000
2 138 93 339000
3 142 92 337000
4 146 91 335000
5 150 90 333000


### Break 3: In your S4 threshold sweep, push the sweep 20 steps further than the minimum.

In [51]:
fp, fn = 130, 95
cost_fp, cost_fn = 300, 3200
for step in range(94, 116):
    f_p = fp + 4*step
    f_n = fn - step
    total = (f_p * cost_fp) + (f_n * cost_fn)
    print(step, f_p, f_n, total)

94 506 1 155000
95 510 0 153000
96 514 -1 151000
97 518 -2 149000
98 522 -3 147000
99 526 -4 145000
100 530 -5 143000
101 534 -6 141000
102 538 -7 139000
103 542 -8 137000
104 546 -9 135000
105 550 -10 133000
106 554 -11 131000
107 558 -12 129000
108 562 -13 127000
109 566 -14 125000
110 570 -15 123000
111 574 -16 121000
112 578 -17 119000
113 582 -18 117000
114 586 -19 115000
115 590 -20 113000


**Exact error or behavior I observed:**

```
94 506 1 155000
95 510 0 153000
96 514 -1 151000
97 518 -2 149000
98 522 -3 147000
99 526 -4 145000
100 530 -5 143000
101 534 -6 141000
102 538 -7 139000
103 542 -8 137000
104 546 -9 135000
105 550 -10 133000
106 554 -11 131000
107 558 -12 129000
108 562 -13 127000
109 566 -14 125000
110 570 -15 123000
111 574 -16 121000
112 578 -17 119000
113 582 -18 117000
114 586 -19 115000
115 590 -20 113000
```

**What this tells me about how it works:**

> The minimum number of possible false negatives is 0 which starts at step 95. To push the sweep 20 past that minumum, I set the range to end at 116. The error I observed shows that the formula doesnt account for realistic limitations. Instead, it continues to subtract past zero false negatives, resulting in negative occurances of false negatives (which is not possible) and this causes the model to calculate an unrealistically low total cost error. 

## 3. Results

**Your hand computed metrics for one scenario**

> Enterprise spam filter:
```
Accuracy = (1200 + 48000) / (1200 + 95 + 310 + 48000) 
         ≈ 0.991836

Recall = 1200 / (1200 + 310)
       ≈ 0.794702

Precision = 1200 / (1200 + 95)
          ≈ 0.926641

F1 = 2(0.926641)(0.794702) / (0.926641 + 0.79470)
   ≈ 0.855615
```

**Total error cost for all four scenarios**

```scenario_id  accuracy  total_error_cost  
0          S1    0.9900            500008  
3          S4    0.9536            343000  
2          S3    0.9930             27140  
1          S2    0.9918              7520 
```
**Your four metric calls**

> S1, Hospital sepsis screening:  
I'd optimize recall. This choice is driven by the substantially higher cost per false negative `$250000` compared to the price of a false positive $1. This shows the damage missing a real case could do.  

> S2, Enterprise spam filter:
I'd optimize precision. The cost per false positive `$40` compared to the price of a false negative `$12` means flagging an email as spam incorrectly, is the most expensive error in the scenario.  

> S3, Warehouse part defect scan:
I'd optimize recall. The cost per false negative `$900` outweighs the price of a false positive `$85` so avoiding a missed detection should be prioritized.  

> S4, Loan pre-approval triage:
I'd optimize recall. The cost per false negative `$3200` compared to the price of a false positive `$300` makes risk detetction for risk cutomers, the most important factor.  

**Your S4 cost minimizing threshold and the saving**
> The current baseline model (step 0) gives the baseline cost of `$39,000` with 130 false positives, plus 95 false negatives at `$304,000`, totaling `$343,000`. However if there is a zero-default policy (step 95), the model eliminates these 95 false negatives, reducing them to O. In exchange, it adds 380 false positives (95 steps × 4), bringing the total false positives to 510. Multiplying 510 by the `$300` has a total error cost of `$153,000`. Therefore havine a a zero-default policy is a far better and cost effective model, saving `$190,000`.

## 4. Reflection

Your own words. No AI assistance. Graders check these against the outputs in your notebook.

### 1. Report your Step 3 prediction and what the table actually showed. Quote your own accuracy and total error cost figures for at least two scenarios in your answer.

> I predicted that the model with the highest accuracy would have the lowest total error costs. However, the table showed that the models accuracy had no correlation with its actual financial damage when the cost of errors are imbalanced. As shown, scenario 1 Hospital sepsis screening (S1) had a high accuracy of 0.99 but also had a high total error cost of `$500,008`. Whereas scenario 2, Enterprise spam filter (S2) had almost the same accuracy of 0.9918, but it's total error. Cost is just `$7,520`. Comparing these two scenarios' accuracy and total error cost, proves that when one type of error is extremely expensive you can't rely on a model's accuracy to measure real world use cases. 

### 2. Scenario S1 has an accuracy of 0.99 in the file. Explain, using the actual counts, why that number is close to meaningless and what you would report to a hospital administrator instead.

> In the CSV file, Hospital sepsis screening (S1) had an accuracy of 0.99. However, using the actual accounts, this number is close to meaningless, because it hides the the severity of mistakes behind correct predictions. Out of 1000 patients it recorded 8 false positives and 2 false negatives. Hospital cases do not hold equal weights. A single false negative not only cost `$250,000`, but also effects human lives, family, health and the chance to be correctly treated. Rather than accuracy, recall, and the total number of false negatives should be reported to a hospital administrator. This way, they'd understand the impact of missing even just one life-threatening case.

### 3. Your S4 threshold sweep found a minimum. State the setting and the dollar saving against the starting point. Then say what would have to be true operationally for that setting to be a bad idea anyway.

> The minimum total error cost is reached at step 95. This setting saves `$190,000` compared to starting baseline at step 0. Operationally, starting at step 95 would be a bad idea if a bank couldn't  sufficiently process the volume of 510 false positives. It will lead to delays and frustrated qualifying applicants who'd possibly take their business to another bank. What's more, this bank's reputation would be damaged, potentially being more expensive than the total error cost. 

### 4. In the second break it exercise you equalized all the costs. Explain what that did to the rankings and what it tells you about where the real information in this lab is coming from.

> When I set all four scenarios to have equal false positive and false negative costs, Hospital sepsis screening (S1) went from being the most expensive cost of error to the least expensive. This shows that the total_error_cost would only account for the number of errors not the financial weight of the error, let alone the critical damage to someone's health. This proves that real information in this lab comes directly from a business cost matrix not machine learning error counts. A model it shouldn't be valued by its accuracy, but by the cost of weighted mistakes.

## 5. AI use disclosure
Required by the course AI Use Policy. Write none if you used no AI.

- **Tool used:** ChatGPT
- **What I asked for:** 

> To explain the threshold sweep. 

- **What I used from it:** 

> The concept and explanation. 

- **How I verified it:** 

> I adjusted the parameters provided in the lab`s step 5 instructions

## 6. Checklist

- [x] Predictions written before running
- [x] Actual results recorded, including the ones I got wrong
- [x] Exact error text captured for every break it item
- [x] Notebook runs top to bottom with outputs visible
- [x] PDF exported after the full run
- [x] Reflection written by me
- [x] Files uploaded individually, no archive